# Reproduce figures & Table 1

Regenerates every artifact for a run. Uses the **mock provider** offline by default; point
`CONFIG` at a real run YAML (with models/keys/`DATABASE_URL` set) to reproduce with live models.

Reproduction of Akata et al. (2025), *Playing repeated games with large language models*.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

from llmgames.config.loader import load_run_spec
from llmgames.loop.tournament import run_tournament
from llmgames.results import generate_results_md

CONFIG = '../config/runs/paper_default.yaml'
MOCK = True  # set False to use the real models in CONFIG

run = load_run_spec(CONFIG)
if MOCK:
    run = run.model_copy(update={
        'models': [m.model_copy(update={'provider': 'mock'}) for m in run.models],
        'cache': run.cache.model_copy(update={'backend': 'memory'}),
        'output_dir': 'nb_results',
    })
result = run_tournament(run)
print('matches:', len(result.results))

## Table 1 — score ratio across families

In [ ]:
from llmgames.metrics import score_ratio
score_ratio.score_ratio_table(result.results)

## Prisoner's Dilemma — defection heat map & cooperation trajectory

In [ ]:
import matplotlib.pyplot as plt
from llmgames.metrics import pd_metrics
pd_results = [r for r in result.results if r.game.name == "Prisoner's Dilemma"]
display(pd_metrics.defection_rate_matrix(pd_results))
traj = pd_metrics.cooperation_trajectory(pd_results)
for player, sub in traj.groupby('player'):
    plt.plot(sub['round'], sub['cooperation_rate'], marker='o', label=player)
plt.xlabel('round'); plt.ylabel('cooperation rate'); plt.legend(fontsize=7); plt.title('PD cooperation'); plt.show()

## Battle of the Sexes — collaboration & P1-preferred

In [ ]:
from llmgames.metrics import bos_metrics
bos_results = [r for r in result.results if r.game.name == 'Battle of the Sexes']
display(bos_metrics.collaboration_rate_matrix(bos_results))
display(bos_metrics.preferred_frequency_matrix(bos_results))

## Human study (Fig 7) — real 195-participant data

In [ ]:
from llmgames.metrics import human
try:
    display(human.summarize_human(human.load_human_data('../data/human/repgames.csv')))
except FileNotFoundError as exc:
    print(exc)

## Full summary document

In [ ]:
print('Wrote', generate_results_md(run, result))